# FLS Inverse: Algorithm Walkthrough

The Faddeev–LeVerrier–Souriau (FLS) algorithm computes the inverse of a multivector
$A \in Cl(d)$ by building a sequence of scalar invariants and multivectors whose
final ratio gives $A^{-1}$.

This notebook walks through the even/odd specialisation and the general FLS inverse
as implemented in `clifford.inverse.fls`.

In [1]:
from clifford.context import Cl
from clifford.multivector import Accum
from clifford.inverse import (
    fls_inverse,
    even_inverse,
    odd_inverse,
    sparse_fls_inverse,
)
import clifford.context as ctx
import numpy as np

## Even-graded multivectors

A multivector is *even-graded* if all odd-grade blades are zero.  Its inverse
is also even-graded, and `even_inverse` exploits this to skip half the
coefficient slots.

In [2]:
Cl(6)

A = Accum()
A.random()
for i in range(len(A.Reg)):
    if ctx.Grade[i] % 2 != 0:
        A.Reg[i] = 0.0

print("Even-graded A (odd blades zeroed):")
print(A)

Even-graded A (odd blades zeroed):
00000000      -1.11412255
00000011       0.78997689
00000101       0.08287637
00000110      -0.15967963
00001001      -0.06139140
00001010      -1.05907288
00001100      -0.07636973
00001111      -0.59263264
00010001       1.20127677
00010010       0.34198004
00010100       0.71778744
00010111       0.32877379
00011000       0.33559246
00011011       0.00303501
00011101       0.51856839
00011110      -0.64575677
00100001       0.47549115
00100010       0.70970326
00100100      -0.37177083
00100111       1.54852281
00101000      -0.72730669
00101011      -0.57500439
00101101      -1.23138925
00101110      -0.96159349
00110000      -0.32329148
00110011      -1.74031080
00110101      -0.91539861
00110110       1.02089522
00111001       0.46773514
00111010       0.45476389
00111100      -0.98128030
00111111      -0.46024620



In [3]:
iA = even_inverse(A)
check = A * iA
print(f"Scalar component:  {check.Reg[0]:.15f}")
print(f"Max off-scalar:    {max(abs(check.Reg[1:])):.2e}")

Scalar component:  0.999999999999999
Max off-scalar:    1.33e-15


## Odd-graded multivectors

Similarly, `odd_inverse` handles multivectors supported entirely on odd-grade blades.

In [4]:
B = Accum()
B.random()
for i in range(len(B.Reg)):
    if ctx.Grade[i] % 2 != 1:
        B.Reg[i] = 0.0

iB = odd_inverse(B)
check = B * iB
print(f"Scalar component:  {check.Reg[0]:.15f}")
print(f"Max off-scalar:    {max(abs(check.Reg[1:])):.2e}")

Scalar component:  1.000000000000022
Max off-scalar:    9.19e-13


## General FLS inverse

`fls_inverse` handles an arbitrary multivector by routing through even/odd
sub-cases and combining.

In [5]:
C = Accum()
C.random()   # general multivector: both even and odd grades populated

iC = fls_inverse(C)
check = C * iC
print(f"fls_inverse — scalar: {check.Reg[0]:.15f},  max off-scalar: {max(abs(check.Reg[1:])):.2e}")

fls_inverse — scalar: 1.000000000000000,  max off-scalar: 9.37e-15


## Sparse FLS

`sparse_fls_inverse` uses the same algorithm but operates on sparse coefficient
vectors, which is faster when many blades are zero.  The results are numerically
identical.

In [6]:
iC_sparse = sparse_fls_inverse(C)
diff = max(abs(iC.Reg - iC_sparse.Reg))
print(f"Max difference between fls and sparse_fls results: {diff:.2e}")

Max difference between fls and sparse_fls results: 9.09e-16


## Residual comparison across dimensions

In [7]:
print(f"{'dim':>4}  {'fls residual':>14}  {'sparse_fls residual':>20}")
print("-" * 44)
for d in [6, 7, 8, 9, 10]:
    Cl(d)
    A = Accum(); A.random()
    r_fls    = max(abs((A * fls_inverse(A)).Reg[1:]))
    r_sparse = max(abs((A * sparse_fls_inverse(A)).Reg[1:]))
    print(f"  {d:2d}    {r_fls:14.2e}    {r_sparse:20.2e}")

 dim    fls residual   sparse_fls residual
--------------------------------------------
   6          1.10e-14                7.41e-15
   7          5.74e-15                3.92e-15
   8          8.67e-14                1.82e-14
   9          3.19e-13                1.25e-13
  10          7.27e-11                1.90e-11
